# ABP Waveform Feature Extraction
Extracts per-patient cardiovascular features from ABP waveforms using a two-element Windkessel model.
Standalone — no InfluxDB/Redis required.

In [6]:
import os
import numpy as np
import pandas as pd
from glob import glob
from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_percentage_error
from numba import njit
from biosppy.signals import ppg

# --- Config ---
DATA_DIR    = "patient_data_abp"
FREQ_HZ     = 125
WEIGHT_KG   = 18.5          # placeholder — no weight data available
MAX_SAMPLES = 7 * 3600 * 125  # first 7 hours at 125 Hz = 3,150,000 samples

## Windkessel Model — Core Math
Adapted from `src-SV/sv_functions.py` (InfluxDB writes removed).

In [7]:
@njit
def make_time_arrays(systolic_period, time, sample_period):
    split_idx = np.searchsorted(time, systolic_period)
    residual_below = systolic_period - time[split_idx - 1]
    residual_above = time[split_idx] - systolic_period

    if time[split_idx] == systolic_period:
        if split_idx == time.size - 1:
            split_idx -= 1
        systolic_times  = time[:split_idx + 1]
        diastolic_times = time[split_idx + 1:] - systolic_times[-1]
    elif residual_below < residual_above:
        systolic_times  = time[:split_idx]
        diastolic_times = time[split_idx:] - time[split_idx]
    else:
        systolic_times  = time[:split_idx]
        diastolic_times = time[split_idx:] - time[split_idx] + sample_period

    return systolic_times, diastolic_times


@njit
def abp_function_loss(x, true_waveform_values, time_array, diastolic_pressure, mean_pressure, period, sample_period):
    sys_t, dia_t = make_time_arrays(x[1], time_array, sample_period)
    sys_p = diastolic_pressure * np.exp(-sys_t / x[0]) + mean_pressure * (period / x[1]) * (1 - np.exp(-sys_t / x[0]))
    dia_p = diastolic_pressure * np.exp(-((dia_t + x[1]) / x[0])) + mean_pressure * (period / x[1]) * np.exp(-dia_t / x[0]) * (1 - np.exp(-x[1] / x[0]))
    return np.sum(np.abs(true_waveform_values - np.concatenate((sys_p, dia_p))))


def check_bounds(beat_vals, systolic_peak_idx, bpm, bpm_available, freq_hz,
                 systolic_vals_fraction=0.5, bpm_decrease_tolerance=0.25, bpm_increase_tolerance=0.5):
    if beat_vals[:systolic_peak_idx].size > systolic_vals_fraction * beat_vals.size:
        return False
    if bpm_available:
        hr = (freq_hz * 60) / beat_vals.size
        lower, upper = bpm * (1 - bpm_decrease_tolerance), bpm * (1 + bpm_increase_tolerance)
        return lower < hr < upper
    return freq_hz * 0.25 < len(beat_vals) <= freq_hz * 2


def remove_outliers_std(data, num_std=2):
    arr = np.array(data, dtype=float)
    m, s = np.mean(arr), np.std(arr)
    return arr[(arr >= m - num_std * s) & (arr <= m + num_std * s)]

In [8]:
def extract_features(abp_values, abp_times, freq_hz=125, weight=18.5, patient_id=None):
    """Extract cardiovascular features from ABP waveform using 2-element Windkessel model."""
    sample_period = 1 / freq_hz

    # Detect beat onsets using biosppy
    try:
        onsets = ppg.find_onsets_kavsaoglu2016(
            signal=abp_values, sampling_rate=freq_hz
        )['onsets']
    except Exception:
        return None

    if len(onsets) < 3:
        return None

    RC_list, TS_list, SV_list, CO_list = [], [], [], []
    PP_list, SBP_list, DBP_list, MAP_list, HR_list, errors = [], [], [], [], [], []

    for i in range(len(onsets) - 1):
        beat = abp_values[onsets[i]:onsets[i + 1] + 1]
        if len(beat) < 3:
            continue

        sys_p   = np.max(beat)
        sys_idx = np.argmax(beat)
        dia_p   = beat[0]

        if not check_bounds(beat, sys_idx, 73, False, freq_hz):
            continue

        time_arr = np.arange(0, beat.size * sample_period, sample_period)[:beat.size]
        period   = time_arr[-1]
        mean_p   = np.mean(beat)

        try:
            res = minimize(
                fun=abp_function_loss,
                x0=np.array([1.5, 0.1]),
                method='Nelder-Mead',
                args=(beat, time_arr, dia_p, mean_p, period, sample_period),
                bounds=[(1e-6, 10), (1e-6, period * 0.5)]
            )
        except Exception:
            continue

        RC, TS = res.x[0], res.x[1]
        hr = (freq_hz * 60) / beat.size
        pp = sys_p - dia_p
        co = (mean_p / RC) * 0.056 * weight * 60
        sv = co / hr

        # MAPE of Windkessel fit
        try:
            sys_t, dia_t = make_time_arrays(TS, time_arr[:-1], sample_period)
            sys_calc = dia_p * np.exp(-sys_t / RC) + mean_p * (period / TS) * (1 - np.exp(-sys_t / RC))
            dia_calc = (dia_p * np.exp(-((dia_t + TS) / RC))
                        + mean_p * (period / TS) * np.exp(-dia_t / RC) * (1 - np.exp(-TS / RC)))
            calc = np.concatenate((sys_calc, dia_calc))
            mape = mean_absolute_percentage_error(beat[:-1], calc) * 100
        except Exception:
            mape = np.nan

        RC_list.append(RC);  TS_list.append(TS)
        SV_list.append(sv);  CO_list.append(co)
        PP_list.append(pp);  SBP_list.append(sys_p)
        DBP_list.append(dia_p); MAP_list.append(mean_p)
        HR_list.append(hr);  errors.append(mape)

    if len(RC_list) == 0:
        return None

    # PPV — pulse pressure variation
    pp_clean = remove_outliers_std(PP_list)
    ppv = (round(((np.max(pp_clean) - np.min(pp_clean)) / np.mean(pp_clean)) * 100, 2)
           if len(pp_clean) > 2 else np.nan)

    # SVV — stroke volume variation
    sv_clean = remove_outliers_std(SV_list)
    svv = (round(((np.max(sv_clean) - np.min(sv_clean)) / np.mean(sv_clean)) * 100, 2)
           if len(sv_clean) > 2 else np.nan)

    shock_idx = round(np.mean(HR_list) / np.mean(SBP_list), 4) if SBP_list else np.nan

    return {
        'patient_id':  patient_id,
        'n_beats':     len(RC_list),
        'RC_mean':     round(np.mean(RC_list),  4),
        'RC_std':      round(np.std(RC_list),   4),
        'TS_mean':     round(np.mean(TS_list),  4),
        'TS_std':      round(np.std(TS_list),   4),
        'SV_mean':     round(np.mean(SV_list),  3),
        'CO_mean':     round(np.mean(CO_list),  3),
        'SBP_mean':    round(np.mean(SBP_list), 2),
        'DBP_mean':    round(np.mean(DBP_list), 2),
        'MAP_mean':    round(np.mean(MAP_list), 2),
        'PP_mean':     round(np.mean(PP_list),  3),
        'HR_mean':     round(np.mean(HR_list),  2),
        'PPV':         ppv,
        'SVV':         svv,
        'Shock_Index': shock_idx,
        'MAPE_mean':   round(np.nanmean(errors), 4),
    }


## Run on All Patients

In [9]:
patient_dirs = sorted(glob(os.path.join(DATA_DIR, "*")))
print(f"Found {len(patient_dirs)} patient folders")

results = []

for patient_dir in patient_dirs:
    patient_id = os.path.basename(patient_dir)
    wf_path = os.path.join(patient_dir, "ABP_125.csv")

    if not os.path.exists(wf_path):
        print(f"  [{patient_id}] No ABP_125.csv — skipping")
        continue

    try:
        df_abp = pd.read_csv(wf_path)
        abp_values = df_abp.iloc[:MAX_SAMPLES, 1].to_numpy(dtype=float).copy()
        abp_times  = df_abp.iloc[:MAX_SAMPLES, 0].to_numpy(dtype=float).copy()
    except Exception as e:
        print(f"  [{patient_id}] Read error: {e}")
        continue

    # Replace negatives with NaN then interpolate
    abp_values[abp_values < 0] = np.nan
    if np.all(np.isnan(abp_values)):
        print(f"  [{patient_id}] All NaN — skipping")
        continue
    nans = np.isnan(abp_values)
    x = np.arange(len(abp_values))
    abp_values[nans] = np.interp(x[nans], x[~nans], abp_values[~nans])

    print(f"  [{patient_id}] Processing {len(abp_values)} samples...", end=" ")
    feat = extract_features(abp_values, abp_times, freq_hz=FREQ_HZ, weight=WEIGHT_KG, patient_id=patient_id)
    if feat:
        results.append(feat)
        print(f"OK — {feat['n_beats']} beats, RC={feat['RC_mean']}, SV={feat['SV_mean']}")

print(f"\nDone. {len(results)}/{len(patient_dirs)} patients processed successfully.")

Found 145 patient folders
  [10003] Processing 2665362 samples... OK — 8 beats, RC=2.0735, SV=14.239
  [10009] Processing 1891936 samples... OK — 3041 beats, RC=1.0362, SV=54.803
  [10011] Processing 2750366 samples... OK — 19 beats, RC=2.494, SV=29.104
  [10013] Processing 27943 samples...   [10015] Processing 3150000 samples... 

KeyboardInterrupt: 

## Results

In [ ]:
features_df = pd.DataFrame(results).set_index("patient_id")
print(features_df.shape)
features_df

(140, 16)


,n_beats,RC_mean,RC_std,TS_mean,TS_std,SV_mean,CO_mean,SBP_mean,DBP_mean,MAP_mean,PP_mean,HR_mean,PPV,SVV,Shock_Index,MAPE_mean
patient_id,,,,,,,,,,,,,,,,
10003,8,2.0735,1.0353,0.1835,0.2618,14.239,5.543920e+02,47.44,15.66,2.57,31.773,41.01,249.62,292.71,0.8646,5.321800e+01
10009,3041,1.0362,0.7251,0.1103,0.0313,54.803,6.350473e+03,105.86,65.05,85.11,40.805,113.19,106.07,85.87,1.0692,5.609000e+00
10011,19,2.4939,3.0298,0.0985,0.0837,29.104,1.705610e+03,70.31,19.32,4.97,50.990,63.09,193.08,382.01,0.8973,2.866495e+14
10015,6855,0.8506,0.1628,0.0950,0.0254,63.068,7.466635e+03,167.47,75.44,100.99,92.030,118.26,19.73,39.26,0.7062,7.761000e+00
10017,1857,1.3438,0.5900,0.1264,0.0261,47.177,3.657587e+03,91.94,58.30,75.24,33.639,78.19,88.94,70.91,0.8504,4.813000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10322,85,1.4982,1.7634,0.1110,0.0640,64.785,6.605635e+03,138.84,87.02,102.02,51.825,98.19,43.07,72.37,0.7072,7.881000e+00
10323,1875,1.1427,0.1045,0.1119,0.0105,34.776,3.364280e+03,82.01,48.21,61.18,33.803,96.55,55.38,63.45,1.1772,3.104000e+00
10329,5,6.0450,3.7490,0.0432,0.0530,1.302,8.520400e+01,5.05,3.69,4.76,1.362,77.80,266.00,212.84,15.4055,7.956000e+00


In [ ]:
features_df.describe().round(3)

,n_beats,RC_mean,RC_std,TS_mean,TS_std,SV_mean,CO_mean,SBP_mean,DBP_mean,MAP_mean,PP_mean,HR_mean,PPV,SVV,Shock_Index,MAPE_mean
count,140.000,140.000,140.000,140.000,140.000,140.000,1.400000e+02,140.000,140.000,140.000,140.000,140.000,140.000,140.000,140.000,1.400000e+02
mean,2126.664,2.371,1.637,0.090,0.049,1648.240,2.073426e+05,93.556,46.233,60.693,47.323,80.236,128.613,183.911,1.555,3.632910e+15
std,4854.687,1.822,1.034,0.041,0.040,16751.528,2.266340e+06,31.125,20.290,32.201,19.177,28.531,105.071,361.661,6.005,3.431099e+16
min,3.000,0.417,0.075,0.005,0.006,0.420,2.730100e+01,0.860,0.080,0.600,0.109,34.680,8.330,17.620,0.245,2.148000e+00
25%,36.500,1.076,0.822,0.065,0.026,28.339,1.355297e+03,79.248,38.002,50.793,37.528,54.400,51.230,80.792,0.653,8.619000e+00
50%,275.500,1.720,1.565,0.097,0.037,42.075,3.553087e+03,96.645,46.245,64.470,47.134,77.965,95.320,142.495,0.937,1.671600e+01
75%,1145.500,3.217,2.218,0.112,0.053,56.457,5.993010e+03,108.830,56.980,76.992,57.758,105.720,177.892,204.572,1.175,1.320516e+13
max,25972.000,9.250,4.195,0.281,0.262,196374.459,2.677600e+07,203.270,115.870,214.410,92.030,130.520,745.820,4296.710,70.194,4.016510e+17


In [ ]:
# Save to CSV for later use
features_df.to_csv("abp_features.csv")
print("Saved to abp_features.csv")

Saved to abp_features.csv
